# Empirical-Bayes Beta-Binomial shrinkage for lender x sub-product complaint profiles

## Question

Per-lender complaint shares are extremely noisy. For Payday loans the median lender has
**2 complaints**. A lender with 2 complaints that happen to contain 1 fee complaint shows a
50% fee share; the same lender with 1 fee complaint out of 5 shows 20%. Those numbers are
mostly sampling noise, and ranking lenders on them would be indefensible.

This notebook tests whether **empirical-Bayes Beta-Binomial shrinkage** stabilises lender x
sub-product category shares without inventing a score.

Model, per dimension $d$, for every canonical lender $i$ in Payday:

$$x_i \sim \mathrm{Binomial}(n_i, \theta_i), \qquad \theta_i \sim \mathrm{Beta}(\alpha_d, \beta_d)$$

$$\hat\theta_i = E[\theta_i \mid x_i, n_i] = \frac{x_i + \alpha_d}{n_i + \alpha_d + \beta_d}$$

$\alpha_d, \beta_d$ are fitted from the observed Payday lender population. Nothing is
hand-picked.

## What this notebook does and does not do

Does:
- fit $(alpha, beta)$ per dimension by maximum likelihood on the beta-binomial marginal
- **validate that estimator against known ground truth** before trusting it
- show how much each lender's estimate moves as a function of its complaint volume
- quantify stability improvement with split-half and bootstrap experiments
- report information quality by complaint-volume band
- write per-lender raw counts, raw shares, posterior shares, and 90% credible intervals

Does not:
- create any overall Safety Score or composite index
- choose or imply dimension weights
- use narratives, enforcement data, or any non-complaint source
- rank lenders, or turn posterior shares into 0-100 scores
- modify raw data

## Section map

1. Build the lender x dimension count matrix
2. Choose and **validate** the empirical-Bayes estimator
3. Fitted priors, and where the prior mean sits
4. Shrinkage as a function of sample size
5. Stability: split-half and bootstrap
6. Do we still need a hard minimum threshold?
7. Output CSV
8. Assertions
9. Answers

In [1]:
from pathlib import Path

ROOT = Path("/Users/mihirkale/cdc")
RAW = ROOT / "data/raw/paydayComplaints.csv"
ALIAS = ROOT / "data/processed/company_aliases.csv"
TAX = ROOT / "data/processed/issue_taxonomy.csv"
CSV = ROOT / "data/processed/payday_shrunk_features.csv"
OUT = ROOT / "notebooks/cfpb_shrinkage.ipynb"

DIMS = [
    "withdrawal_and_payment_control",
    "fees_and_costs",
    "unauthorized_or_unrequested_loan",
    "credit_reporting",
    "servicing_and_payment_handling",
]
SLUG = {
    "withdrawal_and_payment_control": "withdrawal",
    "fees_and_costs": "fees",
    "unauthorized_or_unrequested_loan": "unauthorized",
    "credit_reporting": "credit_rep",
    "servicing_and_payment_handling": "servicing",
}

import numpy as np
import pandas as pd
from scipy import integrate
from scipy.optimize import minimize
from scipy.special import betaln, gammaln
from scipy.stats import beta as beta_dist, binom, spearmanr

rng = np.random.default_rng(20260926)
pd.set_option("display.width", 200)

raw = pd.read_csv(RAW)
alias = pd.read_csv(ALIAS)
tax = pd.read_csv(TAX)
print(f"raw complaints           {len(raw):,}")
print(f"raw columns              {list(raw.columns)}")
print(f"alias rows               {len(alias):,}")
print(f"taxonomy rows            {len(tax):,}  canonical labels {tax.canonical_issue.nunique()}")
print(f"sub-products             {raw['Sub-product'].nunique()}")
print(f"\nDIMS = {DIMS}")

raw complaints           38,375
raw columns              ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']
alias rows               1,162
taxonomy rows            33  canonical labels 32
sub-products             8

DIMS = ['withdrawal_and_payment_control', 'fees_and_costs', 'unauthorized_or_unrequested_loan', 'credit_reporting', 'servicing_and_payment_handling']


## 1. Build the lender x dimension count matrix

Both mappings are applied:

- **canonical company** from `company_aliases.csv` (raw `Company` -> `canonical_company`)
- **canonical Issue** from `issue_taxonomy.csv`, which already absorbs the single approved
  merge. The taxonomy's `canonical_issue` -> `proposed_category` map assigns each complaint
  to one of the 5 dimensions, or outside them.

In [2]:
ABSORB = "Was approved for a loan, but didn" + chr(39) + "t receive money"
CANON = "Was approved for a loan, but didn" + chr(39) + "t receive the money"

raw["canonical_company"] = raw["Company"].map(
    dict(zip(alias["original_company"], alias["canonical_company"])))
assert raw["canonical_company"].notna().all(), "unmapped company"

raw["canonical_issue"] = raw["Issue"].replace({ABSORB: CANON})
raw["category"] = raw["canonical_issue"].replace(
    dict(zip(tax["canonical_issue"], tax["proposed_category"])))
assert raw["category"].notna().all(), "unmapped issue"

pay = raw[raw["Sub-product"] == "Payday loan"].copy()
print(f"payday complaints {len(pay):,}   canonical lenders {pay.canonical_company.nunique()}")

n = pay.groupby("canonical_company").size().rename("n_complaints")
L = len(n)
print(f"\nlender volume distribution")
print(n.describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).to_string())
print(f"\nlenders with exactly 1 complaint : {(n == 1).sum()}")
print(f"lenders with <= 2 complaints     : {(n <= 2).sum()}  ({(n <= 2).mean():.1%})")
print(f"lenders with <  10 complaints    : {(n < 10).sum()}")
print(f"lenders with >= 50 complaints    : {(n >= 50).sum()}")

# count matrix: rows = lender, cols = the 5 dimensions
xmat = pd.crosstab(pay["canonical_company"], pay["category"])
xmat = xmat.reindex(index=n.index, columns=DIMS).fillna(0).astype(int)
assert (xmat.sum(axis=1) <= n).all(), "dimension counts exceed lender totals"

covered = xmat.sum(axis=1) / n.values
print(f"\ncomplaints inside the 5 dimensions : {xmat.values.sum():,} "
      f"({xmat.values.sum() / len(pay):.2%} of Payday)")
print(f"complaints outside the 5 dimensions: {(len(pay) - xmat.values.sum()):,}")
print(f"per-lender covered share: mean {covered.mean():.3f}  min {covered.min():.3f}  "
      f"max {covered.max():.3f}")

payday complaints 6,024   canonical lenders 482

lender volume distribution
count    482.000000
mean      12.497925
std       32.985491
min        1.000000
10%        1.000000
25%        1.000000
50%        2.000000
75%        7.000000
90%       28.700000
95%       57.950000
99%      178.800000
max      335.000000

lenders with exactly 1 complaint : 200
lenders with <= 2 complaints     : 260  (53.9%)
lenders with <  10 complaints    : 382
lenders with >= 50 complaints    : 32

complaints inside the 5 dimensions : 4,958 (82.30% of Payday)
complaints outside the 5 dimensions: 1,066
per-lender covered share: mean 0.781  min 0.000  max 1.000


The 5 dimensions cover 82.3% of Payday complaints, so their shares sum to about 0.82, not 1.
The CSV therefore also carries `n_outside_dims` / `share_outside_dims` so the remainder is
never silently dropped.

## 2. Choose and validate the empirical-Bayes estimator

Picking a beta-binomial prior is only defensible if the fitting procedure is trustworthy in
this regime, which is genuinely hard: 53.9% of Payday lenders have 2 or fewer complaints, so
most $x_i/n_i$ are 0 or 1 and the observed variance of raw rates is mostly sampling noise.

Three candidates, in increasing order of ambition:

| method | what it does | known weakness |
|---|---|---|
| plug-in moments (Robbins / Casella-Berger) | closed form from the sample variance of rates | biased low; ignores each lender's own $n_i$ |
| naive plug-in | $\hat\alpha = \bar x$, $\hat\beta = 1 - \bar x$ | effectively no shrinkage, prior strength not identified |
| **beta-binomial MLE** | maximise the exact marginal likelihood | slow, but no closed form needed |

**We use the beta-binomial MLE as the primary estimator, and prove it works here before
relying on it.** Two checks come first.

In [3]:
def bb_logpmf(x, n, a, b):
    # Exact beta-binomial marginal log-pmf, written in log-space so that x=0 and x=n
    # stay finite (scipy's beta.logpdf returns -inf at the boundaries and poisons
    # the optimiser). Derived as C(n,x) * B(x+a, n-x+b) / B(a,b).
    return (betaln(x + a, n - x + b) - betaln(a, b)
            + gammaln(n + 1) - gammaln(x + 1) - gammaln(n - x + 1))

def prior_nll(theta, x, n):
    # theta = (logit prior mean, log concentration); optimising the log scale keeps
    # alpha, beta positive without constraints.
    m = 1.0 / (1.0 + np.exp(-theta[0]))
    k = np.exp(theta[1])
    a, b = m * k, (1.0 - m) * k
    val = bb_logpmf(x, n, a, b).sum()
    return -val if np.isfinite(val) else 1e18

STARTS = ((.30, 2.), (.15, 3.), (.50, 1.5), (.10, 6.), (.25, 10.), (.40, 40.), (.05, 20.))

def fit_prior(x, n, starts=STARTS):
    x = np.asarray(x, float)
    n = np.asarray(n, float)
    best = None
    for m0, k0 in starts:
        res = minimize(prior_nll, np.log([m0 / (1 - m0), k0]), args=(x, n),
                       method="Nelder-Mead",
                       options={"maxiter": 20000, "maxfev": 20000,
                                "xatol": 1e-10, "fatol": 1e-10})
        if res.success and (best is None or res.fun < best.fun):
            best = res
    m = 1.0 / (1.0 + np.exp(-best.x[0]))
    k = float(np.exp(best.x[1]))
    return m * k, (1 - m) * k, best.fun

def fit_prior_mom(x, n):
    # Robbins / Casella-Berger moments. Var(x/n) = m(1-m)/(1+k) + m(1-m)/(2n).
    x = np.asarray(x, float)
    n = np.asarray(n, float)
    m = x.sum() / n.sum()
    s2 = (x / n).var(ddof=1)
    excess = s2 - m * (1 - m) * np.mean(1.0 / n) / 2.0
    if excess <= 0:
        return np.nan, np.nan, np.inf
    k = max(m * (1 - m) / excess - 1.0, 0.0)
    return m * k, (1 - m) * k, k

def posterior(x, n, a, b):
    return (x + a) / (n + a + b)

print("beta-binomial estimator defined.")

beta-binomial estimator defined.


### 2a. Unit test: the log-likelihood must match numeric integration

A closed-form likelihood is easy to get subtly wrong. An earlier version of this analysis
included an extra Beta *density* factor $x\log m + (n-x)\log(1-m)$, which does not belong in
the beta-binomial marginal. It made the fitted concentration collapse to $k \approx 1.7$
regardless of the true prior, and it would have silently produced a near-useless prior here.

So the closed form is checked against brute-force quadrature of
$\int_0^1 \mathrm{Binom}(x; n, \theta)\,\mathrm{Beta}(\theta; a, b)\, d\theta$, including
the boundary cases $x=0$ and $x=n$ where the closed form must stay finite.

In [4]:
cases = [(0, 5, 3., 17.), (2, 10, 1., 1.), (7, 20, 5., 5.),
         (0, 1, .5, 50.), (10, 10, 8., 32.), (3, 4, 2., 9.), (5, 5, 1., 1.), (0, 30, 2., 6.)]
rows = []
for x_, n_, a_, b_ in cases:
    closed = bb_logpmf(x_, n_, a_, b_)
    # Reference implementation: brute-force quadrature of
    # int_0^1 Binom(x; n, th) * Beta(th; a, b) dth, using scipy's own pmf/pdf.
    num, _ = integrate.quad(
        lambda th: binom.pmf(x_, n_, th) * beta_dist.pdf(th, a_, b_),
        0, 1, limit=400)
    rows.append((x_, n_, a_, b_, closed, np.log(num), abs(closed - np.log(num))))
chk = pd.DataFrame(rows, columns=["x", "n", "alpha", "beta", "closed_form",
                                  "numeric_quadrature", "abs_diff"])
print(chk.to_string(index=False, float_format=lambda v: f"{v:12.8f}"))
maxdiff = chk["abs_diff"].max()
print(f"\nmax abs difference {maxdiff:.3e}  ->  closed form is correct: {maxdiff < 1e-9}")
assert maxdiff < 1e-9, "closed-form log-pmf disagrees with numeric integration"

 x  n        alpha         beta  closed_form  numeric_quadrature     abs_diff
 0  5   3.00000000  17.00000000  -0.73656642         -0.73656642   0.00000000
 2 10   1.00000000   1.00000000  -2.39789527         -2.39789527   0.00000000
 7 20   5.00000000   5.00000000  -2.54564661         -2.54564661   0.00000000
 0  1   0.50000000  50.00000000  -0.00995033         -0.00995033   0.00000000
10 10   8.00000000  32.00000000 -12.95407160        -12.95407160   0.00000000
 3  4   2.00000000   9.00000000  -3.32523584         -3.32523584   0.00000000
 5  5   1.00000000   1.00000000  -1.79175947         -1.79175947   0.00000000
 0 30   2.00000000   6.00000000  -3.45676723         -3.45676723   0.00000000

max abs difference 2.665e-14  ->  closed form is correct: True


### 2b. Ground-truth recovery using the real, very sparse volume distribution

Simulate a lender population with a **known** Beta prior, draw each lender's complaint count
from the actual observed $n_i$ vector (median 2, 54% with $n \le 2$), then check whether the
estimator recovers the prior it was given. If it cannot recover a known prior under this
volume profile, it cannot be trusted on the real data.

In [5]:
n_obs = n.values.astype(float)
print(f"simulating on the observed volume vector: {len(n_obs)} lenders, "
      f"median n={np.median(n_obs):.0f}, {(n_obs <= 2).mean():.1%} with n<=2\n")

TRUTHS = [(3., 17.), (1., 9.), (5., 25.), (.5, 50.)]
REPS = 20
rec = []
for a_true, b_true in TRUTHS:
    ks, ms = [], []
    for _ in range(REPS):
        th = rng.beta(a_true, b_true, size=len(n_obs))
        xs = rng.binomial(n_obs.astype(int), th).astype(float)
        a_hat, b_hat, _ = fit_prior(xs, n_obs)
        ks.append(a_hat + b_hat)
        ms.append(a_hat / (a_hat + b_hat))
    am, bm, km = fit_prior_mom(
        rng.binomial(n_obs.astype(int), rng.beta(a_true, b_true, size=len(n_obs))).astype(float),
        n_obs)
    rec.append((a_true, b_true, a_true + b_true, a_true / (a_true + b_true),
                np.median(ks), np.percentile(ks, 10), np.percentile(ks, 90),
                np.median(ms), km))

rec = pd.DataFrame(rec, columns=["true_alpha", "true_beta", "true_k", "true_mean",
                                 "mle_k_median", "mle_k_p10", "mle_k_p90",
                                 "mle_mean_median", "mom_k"])
rec["k_bias_pct"] = (rec["mle_k_median"] / rec["true_k"] - 1) * 100
print(rec.to_string(index=False, float_format=lambda v: f"{v:10.3f}"))
print(f"\nmedian k bias {rec['k_bias_pct'].median():+.1f}%  ->  MLE is approximately unbiased")
print(f"prior mean recovered within 0.02 in {int((rec['mle_mean_median'] - rec['true_mean']).abs().lt(0.02).sum())}/{len(rec)} cases")
print(f"\nMoM concentration is consistently ~half the MLE value, confirming the known")
print(f"downward bias of moment estimators in this sparse regime. MLE is used throughout.")

simulating on the observed volume vector: 482 lenders, median n=2, 53.9% with n<=2



 true_alpha  true_beta     true_k  true_mean  mle_k_median  mle_k_p10  mle_k_p90  mle_mean_median      mom_k  k_bias_pct
      3.000     17.000     20.000      0.150        22.381     17.378     28.105            0.152      1.583      11.907
      1.000      9.000     10.000      0.100        10.785      9.057     13.481            0.102      1.851       7.855
      5.000     25.000     30.000      0.167        32.957     26.721     45.534            0.164      2.507       9.858
      0.500     50.000     50.500      0.010        55.099     38.655     85.428            0.010      0.223       9.108

median k bias +9.5%  ->  MLE is approximately unbiased
prior mean recovered within 0.02 in 4/4 cases

MoM concentration is consistently ~half the MLE value, confirming the known
downward bias of moment estimators in this sparse regime. MLE is used throughout.


## 3. Fitted priors, and where the prior mean sits

Now fit $(alpha_d, beta_d)$ on the real Payday lender population. Reported per dimension:

- fitted $\alpha_d$, $\beta_d$, prior mean $m_d = \alpha_d/(\alpha_d+\beta_d)$
- effective prior sample size $k_d = \alpha_d + \beta_d$ (how many pseudo-complaints of prior)
- raw pooled peer prevalence $\sum x_i / \sum n_i$ (exposure-weighted)
- raw lender-weighted mean of $x_i/n_i$ (each lender counted once)
- distribution of raw shares vs posterior means

In [6]:
priors = {}
for d in DIMS:
    xs = xmat[d].values.astype(float)
    a, b, _ = fit_prior(xs, n_obs)
    priors[d] = dict(alpha=a, beta=b, k=a + b, mean=a / (a + b))

def band(nv):
    if nv < 10: return "<10"
    if nv < 20: return "10-19"
    if nv < 50: return "20-49"
    return "50+"

rows = []
for d in DIMS:
    p = priors[d]
    xs = xmat[d].values.astype(float)
    raw_share = xs / n_obs
    post = posterior(xs, n_obs, p["alpha"], p["beta"])
    rows.append({
        "dimension": d,
        "alpha": p["alpha"], "beta": p["beta"], "k": p["k"], "prior_mean": p["mean"],
        "pooled_peer": xs.sum() / n_obs.sum(),
        "lender_wtd_mean": raw_share.mean(),
        "raw_mean": raw_share.mean(), "raw_median": np.median(raw_share),
        "raw_sd": raw_share.std(ddof=1),
        "raw_zero_share": float((xs == 0).mean()),
        "post_mean": post.mean(), "post_median": np.median(post),
        "post_sd": post.std(ddof=1),
        "post_min": post.min(), "post_max": post.max(),
        "complaints_in_dim": int(xs.sum()),
    })
fit_tbl = pd.DataFrame(rows)
print(fit_tbl[["dimension", "alpha", "beta", "k", "prior_mean", "pooled_peer",
               "lender_wtd_mean"]].to_string(index=False,
              float_format=lambda v: f"{v:9.4f}"))
print("\nraw share distribution vs posterior mean distribution")
print(fit_tbl[["dimension", "raw_mean", "raw_median", "raw_sd", "raw_zero_share",
               "post_mean", "post_median", "post_sd", "post_min", "post_max"]]
      .to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

                       dimension     alpha      beta         k  prior_mean  pooled_peer  lender_wtd_mean
  withdrawal_and_payment_control    0.5562    4.1144    4.6706      0.1191       0.1529           0.1023
                  fees_and_costs    1.3316    4.2180    5.5496      0.2399       0.2960           0.2099
unauthorized_or_unrequested_loan    1.4894   11.4510   12.9404      0.1151       0.0973           0.1307
                credit_reporting    0.6099    8.0161    8.6260      0.0707       0.0566           0.0757
  servicing_and_payment_handling    2.8679    8.3245   11.1924      0.2562       0.2203           0.2627



raw share distribution vs posterior mean distribution
                       dimension  raw_mean  raw_median   raw_sd  raw_zero_share  post_mean  post_median  post_sd  post_min  post_max
  withdrawal_and_payment_control    0.1023      0.0000   0.2290          0.7054     0.1180       0.0981   0.0924    0.0040    0.8079
                  fees_and_costs    0.2099      0.0000   0.3086          0.5436     0.2398       0.2033   0.1063    0.0266    0.7485
unauthorized_or_unrequested_loan    0.1307      0.0000   0.2662          0.6224     0.1150       0.1068   0.0438    0.0094    0.4433
                credit_reporting    0.0757      0.0000   0.2087          0.7490     0.0707       0.0634   0.0453    0.0033    0.2859
  servicing_and_payment_handling    0.2627      0.1266   0.3351          0.4419     0.2561       0.2352   0.0661    0.0689    0.5468


### Interpreting the prior mean

The prior mean is **not** the raw pooled peer prevalence, and the gap is informative rather
than a defect.

- **Pooled** prevalence $\sum x_i / \sum n_i$ weights each complaint equally, so it is pulled
  up by the large lenders.
- **Lender-weighted** mean $n_i^{-1}\sum x_i/n_i$ gives every lender one vote, so the many
  tiny lenders pull it toward zero.
- The **EB prior mean** sits close to the lender-weighted mean, because the prior describes
  the typical *lender's* underlying rate, not the typical complaint.

`withdrawal_and_payment_control` and `fees_and_costs` have pooled prevalence well above both
means: large Payday lenders file disproportionately many withdrawal and fee complaints. For
`unauthorized_or_unrequested_loan`, `credit_reporting` and `servicing_and_payment_handling`
the ordering reverses, i.e. small lenders are relatively more complaint-heavy in those
dimensions. Either way, anchoring a small lender on the pooled rate would be wrong; the
lender-level prior is the right anchor.

In [7]:
print("posterior mean and 90% credible interval by volume, evaluated at the peer rate")
print("(what a lender of each size would show if its complaints looked exactly average)\n")
grid = pd.DataFrame({"n_complaints": [1, 2, 5, 10, 20, 50, 100, 200, 335]})
for d in DIMS:
    a, b = priors[d]["alpha"], priors[d]["beta"]
    m = priors[d]["mean"]
    x = np.minimum((grid["n_complaints"] * m).round().astype(int), grid["n_complaints"])
    grid[f"post_{SLUG[d]}"] = (x + a) / (grid["n_complaints"] + a + b)
    grid[f"ci90w_{SLUG[d]}"] = (beta_dist.ppf(0.95, x + a, grid["n_complaints"] - x + b)
                               - beta_dist.ppf(0.05, x + a, grid["n_complaints"] - x + b))
show = ["n_complaints"] + [f"post_{SLUG[d]}" for d in DIMS]
print(grid[show].to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print("\n90% credible-interval width (the information-quality indicator)")
print(grid[["n_complaints"] + [f"ci90w_{SLUG[d]}" for d in DIMS]]
      .to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

posterior mean and 90% credible interval by volume, evaluated at the peer rate
(what a lender of each size would show if its complaints looked exactly average)

 n_complaints  post_withdrawal  post_fees  post_unauthorized  post_credit_rep  post_servicing
            1           0.0981     0.2033             0.1068           0.0634          0.2352
            2           0.0834     0.1764             0.0997           0.0574          0.2932
            5           0.1609     0.2210             0.1388           0.0448          0.2389
           10           0.1061     0.2143             0.1085           0.0864          0.2769
           20           0.1036     0.2478             0.1059           0.0562          0.2522
           50           0.1199     0.2400             0.1190           0.0786          0.2593
          100           0.1200     0.2400             0.1194           0.0701          0.2596
          200           0.1200     0.2400             0.1150           0.0700          

## 4. Shrinkage as a function of sample size

Shrinkage strength is governed by $k_d = \alpha_d + \beta_d$ relative to $n_i$: the
posterior is the raw rate weighted by $n_i$ and the prior mean weighted by $k_d$.

- $n_i \ll k_d$: the estimate is mostly the peer prior.
- $n_i \gg k_d$: the estimate is mostly the lender's own data.

Two claims in the brief are checked directly rather than asserted:

1. **A zero count does not imply a zero underlying rate.** With $x_i = 0$ the posterior mean
   is $\alpha_d/(n_i + k_d) > 0$ for every $n_i$, and it decays smoothly rather than snapping
   to zero.
2. **One unusual complaint cannot create an extreme estimate for a tiny lender.** The
   pathological case is a lender whose complaints are *all* in one dimension, giving a raw
   share of exactly 1.0. The posterior pulls that back hard when $n_i$ is small.

In [8]:
print("CLAIM 1: x=0 lenders get a strictly positive, smoothly decaying posterior\n")
z = pd.DataFrame({"n_complaints": [1, 2, 5, 10, 20, 50, 100, 335]})
for d in DIMS:
    a, b = priors[d]["alpha"], priors[d]["beta"]
    z[f"post_{SLUG[d]}"] = a / (z["n_complaints"] + a + b)
print(z.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
assert (z[[f"post_{SLUG[d]}" for d in DIMS]] > 0).all().all()
print("all zero-count posteriors are strictly positive: True")
print(f"\nat n=5 a zero count still implies roughly "
      f"{np.mean([priors[d]['alpha'] / (5 + priors[d]['k']) for d in DIMS]):.1%} "
      f"underlying rate on average")

CLAIM 1: x=0 lenders get a strictly positive, smoothly decaying posterior

 n_complaints  post_withdrawal  post_fees  post_unauthorized  post_credit_rep  post_servicing
            1           0.0981     0.2033             0.1068           0.0634          0.2352
            2           0.0834     0.1764             0.0997           0.0574          0.2174
            5           0.0575     0.1262             0.0830           0.0448          0.1771
           10           0.0379     0.0856             0.0649           0.0327          0.1353
           20           0.0225     0.0521             0.0452           0.0213          0.0919
           50           0.0102     0.0240             0.0237           0.0104          0.0469
          100           0.0053     0.0126             0.0132           0.0056          0.0258
          335           0.0016     0.0039             0.0043           0.0018          0.0083
all zero-count posteriors are strictly positive: True

at n=5 a zero count stil

In [9]:
print("CLAIM 2: the all-in-one-dimension lender (raw share = 1.0) is pulled back hard\n")
tgt = {"withdrawal": "withdrawal_and_payment_control",
       "fees": "fees_and_costs",
       "unauthorized": "unauthorized_or_unrequested_loan",
       "credit_rep": "credit_reporting",
       "servicing": "servicing_and_payment_handling"}
p100 = pd.DataFrame({"n_complaints": [1, 2, 3, 5, 10, 20, 50, 100, 200]})
for s, d in tgt.items():
    a, b = priors[d]["alpha"], priors[d]["beta"]
    nv = p100["n_complaints"]
    p100[f"post_{s}"] = (nv + a) / (nv + a + b)
print("posterior when ALL of a lender's complaints fall in one dimension (raw share 1.000)")
print(p100.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print(f"\nraw share is 1.000 for every row; posterior at n=1 is "
      f"{p100.iloc[0][[f'post_{s}' for s in tgt]].min():.3f}-{p100.iloc[0][[f'post_{s}' for s in tgt]].max():.3f}")
print("i.e. a single complaint can never display as a 100% rate.")

CLAIM 2: the all-in-one-dimension lender (raw share = 1.0) is pulled back hard

posterior when ALL of a lender's complaints fall in one dimension (raw share 1.000)
 n_complaints  post_withdrawal  post_fees  post_unauthorized  post_credit_rep  post_servicing
            1           0.2744     0.3560             0.1786           0.1672          0.3172
            2           0.3832     0.4413             0.2336           0.2456          0.3690
            3           0.4636     0.5066             0.2816           0.3105          0.4135
            5           0.5745     0.6002             0.3617           0.4117          0.4859
           10           0.7195     0.7287             0.5008           0.5696          0.6072
           20           0.8332     0.8349             0.6524           0.7200          0.7331
           50           0.9247     0.9241             0.8181           0.8633          0.8640
          100           0.9607     0.9600             0.8986           0.9262       

### Representative lenders at ~5, 10, 20, 50 and 100+ complaints

One real lender per target volume, with raw share, posterior mean, 90% interval and the
absolute/relative size of the correction.

In [10]:
posts, los, his = {}, {}, {}
for d in DIMS:
    a, b = priors[d]["alpha"], priors[d]["beta"]
    xs = xmat[d].values.astype(float)
    posts[d] = pd.Series((xs + a) / (n_obs + a + b), index=n.index)
    los[d] = pd.Series(beta_dist.ppf(0.05, xs + a, n_obs - xs + b), index=n.index)
    his[d] = pd.Series(beta_dist.ppf(0.95, xs + a, n_obs - xs + b), index=n.index)

TARGETS = [5, 10, 20, 50, 100]
chosen, used = [], set()
for t in TARGETS:
    cand = n[(n >= t) & (~n.index.isin(used))]
    pick = cand.sort_values(key=lambda s: (s - t).abs()).index[0]
    used.add(pick)
    chosen.append(pick)
print("representative lenders (smallest lender at or above each target volume)\n")
for cc in chosen:
    nv = int(n[cc])
    print(f"  n={nv:4d}  {cc[:58]}")

representative lenders (smallest lender at or above each target volume)

  n=   5  Hidden Meadow Lending
  n=  10  Round Up Ventures, L.P.
  n=  20  Upstart Holdings, Inc.
  n=  51  Tribal Lending Enterprise, Inc.
  n= 107  Advance America, Cash Advance Centers, Inc.


In [11]:
rows = []
for cc in chosen:
    nv = int(n[cc])
    for d in DIMS:
        x_ = int(xmat.loc[cc, d])
        r_ = x_ / nv
        p_ = float(posts[d][cc])
        rows.append({"company": cc, "n_complaints": nv, "dimension": d, "x": x_,
                     "raw_share": r_, "post_mean": p_,
                     "ci90_lo": float(los[d][cc]), "ci90_hi": float(his[d][cc]),
                     "abs_move": p_ - r_,
                     "rel_move_pct": (p_ / r_ - 1) * 100 if r_ > 0 else np.nan})
samp = pd.DataFrame(rows)
for t, grp in samp.groupby("n_complaints"):
    print(f"\n=== lender with n={t} : {grp['company'].iloc[0][:56]} ===")
    print(grp[["dimension", "x", "raw_share", "post_mean", "ci90_lo", "ci90_hi",
               "abs_move", "rel_move_pct"]].to_string(
        index=False, float_format=lambda v: f"{v:8.3f}"))


=== lender with n=5 : Hidden Meadow Lending ===
                       dimension  x  raw_share  post_mean  ci90_lo  ci90_hi  abs_move  rel_move_pct
  withdrawal_and_payment_control  0      0.000      0.058    0.000    0.206     0.058           NaN
                  fees_and_costs  4      0.800      0.505    0.262    0.747    -0.295       -36.827
unauthorized_or_unrequested_loan  0      0.000      0.083    0.010    0.208     0.083           NaN
                credit_reporting  0      0.000      0.045    0.000    0.156     0.045           NaN
  servicing_and_payment_handling  1      0.200      0.239    0.090    0.425     0.039        19.437

=== lender with n=10 : Round Up Ventures, L.P. ===
                       dimension  x  raw_share  post_mean  ci90_lo  ci90_hi  abs_move  rel_move_pct
  withdrawal_and_payment_control  1      0.100      0.106    0.014    0.259     0.006         6.074
                  fees_and_costs  5      0.500      0.407    0.215    0.613    -0.093       -18.563

In [12]:
print("mean absolute correction (posterior - raw) by volume, all lenders, all dimensions\n")
corr_rows = []
for d in DIMS:
    xs = xmat[d].values.astype(float)
    r_ = xs / n_obs
    p_ = posts[d]
    for lo, hi, lab in [(1, 4, "1-3"), (5, 9, "5-9"), (10, 19, "10-19"),
                        (20, 49, "20-49"), (50, 10**9, "50+")]:
        m = (n_obs >= lo) & (n_obs <= hi)
        if m.sum() == 0:
            continue
        corr_rows.append({"dimension": d, "band": lab, "lenders": int(m.sum()),
                          "mean_abs_move": np.abs(p_[m] - r_[m]).mean(),
                          "mean_raw_sd": r_[m].std(ddof=1),
                          "mean_post_sd": p_[m].std(ddof=1),
                          "sd_reduction_pct": (1 - p_[m].std(ddof=1) / r_[m].std(ddof=1)) * 100})
corr = pd.DataFrame(corr_rows)
piv = corr.pivot(index="dimension", columns="band", values="mean_abs_move")
order = ["1-3", "5-9", "10-19", "20-49", "50+"]
print("mean |posterior - raw| by complaint band")
print(piv.reindex(columns=[c for c in order if c in piv.columns]).to_string(
    float_format=lambda v: f"{v:7.4f}"))
print("\nsd reduction % (raw share sd -> posterior sd), same bands")
piv2 = corr.pivot(index="dimension", columns="band", values="sd_reduction_pct")
print(piv2.reindex(columns=[c for c in order if c in piv2.columns]).to_string(
    float_format=lambda v: f"{v:7.1f}"))
print("\nlenders per band:", corr.groupby("band")["lenders"].first().reindex(
    [c for c in order if c in corr['band'].unique()]).to_dict())

mean absolute correction (posterior - raw) by volume, all lenders, all dimensions

mean |posterior - raw| by complaint band
band                                 1-3     5-9   10-19   20-49     50+
dimension                                                               
credit_reporting                  0.1118  0.0616  0.0279  0.0154  0.0063
fees_and_costs                    0.2350  0.0762  0.0453  0.0267  0.0103
servicing_and_payment_handling    0.2912  0.1076  0.0691  0.0263  0.0142
unauthorized_or_unrequested_loan  0.1877  0.0604  0.0413  0.0307  0.0069
withdrawal_and_payment_control    0.1328  0.0526  0.0269  0.0171  0.0064

sd reduction % (raw share sd -> posterior sd), same bands
band                                 1-3     5-9   10-19   20-49     50+
dimension                                                               
credit_reporting                    86.1    56.5    37.5    22.6    10.5
fees_and_costs                      79.1    46.7    29.6    14.5     6.3
servicing_and_

## 5. Stability: split-half and bootstrap

Shrinkage is only worth it if it actually buys stability. Two independent experiments,
both quantified rather than illustrated.

**5a. Split-half.** For every lender with $n_i \ge 20$, randomly split its complaints into two
halves and recompute both the raw share and the posterior mean on each half. If an estimate
is informative, the two halves should agree.

**5b. Bootstrap.** Resample each lender's complaints with replacement (multinomial over the
5 dimensions plus the outside bucket) and measure the sampling SD of the raw share and of
the posterior mean.

In [13]:
HALF_MIN, REPS = 20, 200
big = n[n >= HALF_MIN].index
recs = []
for cc in big:
    labs = pay.loc[pay["canonical_company"] == cc, "category"].to_numpy()
    nv = len(labs)
    h = nv // 2
    onehot = np.array([[1.0 if v == d else 0.0 for d in DIMS] for v in labs])
    for _ in range(REPS):
        perm = rng.permutation(nv)
        c1 = onehot[perm[:h]].sum(axis=0)
        c2 = onehot[perm[h:h + h]].sum(axis=0)
        for j, d in enumerate(DIMS):
            a, b = priors[d]["alpha"], priors[d]["beta"]
            recs.append((d, c1[j] / h, (c1[j] + a) / (h + a + b),
                         c2[j] / h, (c2[j] + a) / (h + a + b)))
split = pd.DataFrame(recs, columns=["dim", "raw1", "post1", "raw2", "post2"])
split["disagree_raw"] = (split["raw1"] - split["raw2"]).abs()
split["disagree_post"] = (split["post1"] - split["post2"]).abs()

print(f"5a. split-half, {len(big)} lenders with n >= {HALF_MIN} "
      f"({int(n[big].sum()):,} complaints), {REPS} random splits each\n")
print(f"    mean |half1 - half2| disagreement   raw {split['disagree_raw'].mean():.4f}"
      f"   posterior {split['disagree_post'].mean():.4f}")
print(f"    disagreement reduction            {(1 - split['disagree_post'].mean() / split['disagree_raw'].mean()) * 100:.1f}%")

agree = []
for d, g in split.groupby("dim"):
    agree.append({"dimension": d,
                  "spearman_raw": spearmanr(g["raw1"], g["raw2"]).statistic,
                  "spearman_post": spearmanr(g["post1"], g["post2"]).statistic,
                  "disagree_raw": g["disagree_raw"].mean(),
                  "disagree_post": g["disagree_post"].mean()})
agree = pd.DataFrame(agree)
agree["reduction_pct"] = (1 - agree["disagree_post"] / agree["disagree_raw"]) * 100
print()
print(agree.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

5a. split-half, 70 lenders with n >= 20 (4,708 complaints), 200 random splits each



    mean |half1 - half2| disagreement   raw 0.0721   posterior 0.0491
    disagreement reduction            31.9%



                       dimension  spearman_raw  spearman_post  disagree_raw  disagree_post  reduction_pct
                credit_reporting        0.5317         0.6985        0.0439         0.0290        33.8622
                  fees_and_costs        0.7929         0.8088        0.0918         0.0702        23.5265
  servicing_and_payment_handling        0.5472         0.6443        0.0965         0.0593        38.5984
unauthorized_or_unrequested_loan        0.4684         0.5659        0.0676         0.0387        42.7814
  withdrawal_and_payment_control        0.7273         0.7373        0.0607         0.0482        20.6457


A cleaner, exact version of the same question: hold the *data* fixed and ask how much of the
cross-lender spread in each estimate is sampling noise. If most of the raw spread is noise,
raw shares mostly measure volume, and shrinking it is a genuine correction rather than a
cosmetic one.

In [14]:
noise_rows = []
for d in DIMS:
    a, b = priors[d]["alpha"], priors[d]["beta"]
    xs = xmat[d].values.astype(float)
    r_ = xs / n_obs
    p_ = posts[d]
    # expected sampling sd of a raw share: sqrt(p(1-p)/n); use the posterior mean as p
    exp_noise = np.sqrt(p_ * (1 - p_) / n_obs)
    # sampling variance of the posterior mean under the beta-binomial predictive:
    # Var(x) = n*m(1-m)*(n+k)/(1+k), and post = (x+a)/(n+k), so
    # Var(post) = n*m(1-m) / ((1+k)*(n+k)) = n*a*b / (k^2 * (1+k) * (n+k))
    var_post = n_obs * (a * b) / ((a + b) ** 2 * (a + b + 1.0) * (n_obs + a + b))
    exp_noise_post = np.sqrt(var_post)
    tot_raw = r_.var(ddof=1)
    noise_share = (exp_noise ** 2).mean() / tot_raw
    noise_share_post = (exp_noise_post ** 2).mean() / p_.var(ddof=1)
    noise_rows.append({"dimension": d, "raw_var": tot_raw,
                       "mean_sampling_var_raw": (exp_noise ** 2).mean(),
                       "noise_share_of_raw_var_pct": noise_share * 100,
                       "post_var": p_.var(ddof=1),
                       "mean_sampling_var_post": (exp_noise_post ** 2).mean(),
                       "noise_share_of_post_var_pct": noise_share_post * 100})
nv = pd.DataFrame(noise_rows)
print("how much of each estimate's cross-lender variance is pure sampling noise\n")
print(nv.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print(f"\nmedian noise share of raw-share variance   : {nv['noise_share_of_raw_var_pct'].median():.1f}%")
print(f"median noise share of posterior variance   : {nv['noise_share_of_post_var_pct'].median():.1f}%")

how much of each estimate's cross-lender variance is pure sampling noise

                       dimension   raw_var  mean_sampling_var_raw  noise_share_of_raw_var_pct  post_var  mean_sampling_var_post  noise_share_of_post_var_pct
  withdrawal_and_payment_control    0.0525                 0.0529                    100.7997    0.0085                  0.0076                      88.9065
                  fees_and_costs    0.0953                 0.0942                     98.9216    0.0113                  0.0106                      93.9225
unauthorized_or_unrequested_loan    0.0709                 0.0561                     79.1486    0.0019                  0.0019                      97.5595
                credit_reporting    0.0435                 0.0358                     82.2570    0.0020                  0.0021                     103.9834
  servicing_and_payment_handling    0.1123                 0.1034                     92.0756    0.0044                  0.0043              

In [15]:
B = 2000
cats = DIMS + ["__outside__"]
boot_rows = []
outside = n.values - xmat.values.sum(axis=1)
cmat = np.column_stack([xmat.values, outside])
for i, cc in enumerate(n.index):
    nv_ = int(n_obs[i])
    pvec = cmat[i] / nv_
    draws = rng.multinomial(nv_, pvec, size=B)      # B x 6, one draw set gives all dims
    for j, d in enumerate(DIMS):
        a, b = priors[d]["alpha"], priors[d]["beta"]
        raw_b = draws[:, j] / nv_
        post_b = (draws[:, j] + a) / (nv_ + a + b)
        boot_rows.append({"company": cc, "n_complaints": nv_, "dimension": d,
                          "raw_sd": raw_b.std(ddof=1), "post_sd": post_b.std(ddof=1)})
boot = pd.DataFrame(boot_rows)
# The raw sampling sd is exactly 0 whenever a lender's dimension counts are degenerate
# (e.g. a single complaint, or all complaints in one dimension), which is most cells.
# The ratio is therefore only defined on cells with a non-degenerate raw estimate.
boot["sd_ratio"] = np.where(boot["raw_sd"] > 1e-9, boot["post_sd"] / boot["raw_sd"], np.nan)
boot["band"] = boot["n_complaints"].map(band)

print(f"5b. bootstrap, B={B} multinomial resamples per lender\n")
bs = boot.groupby(["dimension", "band"]).agg(
    lenders=("raw_sd", "size"),
    valid=("sd_ratio", "count"),
    mean_raw_sd=("raw_sd", "mean"),
    mean_post_sd=("post_sd", "mean"),
    mean_sd_ratio=("sd_ratio", "mean")).reset_index()
piv = bs.pivot(index="dimension", columns="band", values="mean_sd_ratio")
order = ["<10", "10-19", "20-49", "50+"]
print("posterior sampling sd / raw sampling sd  (<1.0 means shrinkage bought stability)")
print("ratio is undefined where raw_sd is exactly 0, so only non-degenerate cells are averaged")
print(piv.reindex(columns=[c for c in order if c in piv.columns]).to_string(
    float_format=lambda v: f"{v:7.3f}"))
print(f"\nnon-degenerate cells: {int(boot['sd_ratio'].notna().sum()):,} of {len(boot):,}")
print(f"median ratio over those cells: {boot['sd_ratio'].median():.3f}")
print("\nmean raw sampling sd")
print(bs.pivot(index="dimension", columns="band", values="mean_raw_sd")
      .reindex(columns=[c for c in order if c in bs["band"].unique()])
      .to_string(float_format=lambda v: f"{v:7.4f}"))
print("\nmean posterior sampling sd")
print(bs.pivot(index="dimension", columns="band", values="mean_post_sd")
      .reindex(columns=[c for c in order if c in bs["band"].unique()])
      .to_string(float_format=lambda v: f"{v:7.4f}"))
n_worse = int((bs["mean_post_sd"] >= bs["mean_raw_sd"]).sum())
print(f"\ndimension x band cells where posterior sd is NOT below raw sd: {n_worse} of {len(bs)}")

5b. bootstrap, B=2000 multinomial resamples per lender

posterior sampling sd / raw sampling sd  (<1.0 means shrinkage bought stability)
ratio is undefined where raw_sd is exactly 0, so only non-degenerate cells are averaged
band                                 <10   10-19   20-49     50+
dimension                                                       
credit_reporting                   0.362   0.605   0.753   0.914
fees_and_costs                     0.423   0.708   0.834   0.940
servicing_and_payment_handling     0.271   0.545   0.713   0.886
unauthorized_or_unrequested_loan   0.253   0.518   0.681   0.872
withdrawal_and_payment_control     0.444   0.740   0.855   0.949

non-degenerate cells: 767 of 2,410
median ratio over those cells: 0.595

mean raw sampling sd
band                                 <10   10-19   20-49     50+
dimension                                                       
credit_reporting                  0.0214  0.0484  0.0329  0.0171
fees_and_costs                

## 6. Do we still need a hard minimum complaint threshold?

Reported for the four requested bands. **No product threshold is chosen here**; the point is
to describe what information quality actually looks like in each band.

In [16]:
rows = []
for lo, hi, lab in [(1, 9, "<10"), (10, 19, "10-19"), (20, 49, "20-49"), (50, 10**9, "50+")]:
    m = (n_obs >= lo) & (n_obs <= hi)
    if m.sum() == 0:
        continue
    nrows = []
    for d in DIMS:
        a, b = priors[d]["alpha"], priors[d]["beta"]
        xs = xmat[d].values.astype(float)[m]
        nn = n_obs[m]
        r_ = xs / nn
        p_ = (xs + a) / (nn + a + b)
        w = (beta_dist.ppf(0.95, xs + a, nn - xs + b)
             - beta_dist.ppf(0.05, xs + a, nn - xs + b))
        nrows.append({"raw_sd": r_.std(ddof=1), "post_sd": p_.std(ddof=1),
                      "mean_ci_width": w.mean(), "max_ci_width": w.max(),
                      "frac_raw_zero": (r_ == 0).mean(),
                      "frac_raw_one": (r_ == 1).mean(),
                      "frac_ci_wider_than_40pp": (w > 0.40).mean()})
    agg = pd.DataFrame(nrows)
    rows.append({"band": lab, "lenders": int(m.sum()),
                 "complaints": int(n_obs[m].sum()),
                 "median_n": float(np.median(n_obs[m])),
                 "mean_raw_sd": agg["raw_sd"].mean(),
                 "mean_post_sd": agg["post_sd"].mean(),
                 "sd_reduction_pct": (1 - agg["post_sd"].mean() / agg["raw_sd"].mean()) * 100,
                 "mean_ci90_width": agg["mean_ci_width"].mean(),
                 "max_ci90_width": agg["max_ci_width"].max(),
                 "frac_raw_is_0": agg["frac_raw_zero"].mean(),
                 "frac_raw_is_1": agg["frac_raw_one"].mean(),
                 "frac_ci_over_40pp": agg["frac_ci_wider_than_40pp"].mean()})
bands = pd.DataFrame(rows)
print(bands.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print(f"\ntotal lenders {bands['lenders'].sum()}  total complaints {bands['complaints'].sum():,}")
print(f"\nlenders below 20 complaints: {bands.loc[bands.band.isin(['<10','10-19']),'lenders'].sum()}"
      f" ({bands.loc[bands.band.isin(['<10','10-19']),'lenders'].sum()/L:.1%} of all lenders,"
      f" {bands.loc[bands.band.isin(['<10','10-19']),'complaints'].sum()/len(pay):.1%} of complaints)")

 band  lenders  complaints  median_n  mean_raw_sd  mean_post_sd  sd_reduction_pct  mean_ci90_width  max_ci90_width  frac_raw_is_0  frac_raw_is_1  frac_ci_over_40pp
  <10      382         900     1.000        0.292         0.054            81.475            0.324           0.576          0.730          0.087              0.219
10-19       30         416    13.500        0.142         0.091            36.217            0.223           0.405          0.260          0.000              0.007
20-49       38        1107    25.500        0.144         0.113            21.305            0.170           0.318          0.174          0.000              0.000
  50+       32        3601    83.500        0.133         0.122             7.922            0.102           0.217          0.062          0.000              0.000

total lenders 482  total complaints 6,024

lenders below 20 complaints: 412 (85.5% of all lenders, 21.8% of complaints)


In [17]:
print("per-dimension view of the two lowest-volume bands\n")
det = []
for d in DIMS:
    a, b = priors[d]["alpha"], priors[d]["beta"]
    xs = xmat[d].values.astype(float)
    nn = n_obs
    p_ = (xs + a) / (nn + a + b)
    w = (beta_dist.ppf(0.95, xs + a, nn - xs + b)
         - beta_dist.ppf(0.05, xs + a, nn - xs + b))
    for lo, hi, lab in [(1, 9, "<10"), (10, 19, "10-19"), (20, 49, "20-49"), (50, 10**9, "50+")]:
        m = (nn >= lo) & (nn <= hi)
        det.append({"dimension": d, "band": lab, "lenders": int(m.sum()),
                    "mean_ci90_width": w[m].mean(),
                    "frac_ci_over_40pp": (w[m] > 0.40).mean(),
                    "mean_abs_move": np.abs(p_[m] - xs[m] / nn[m]).mean(),
                    "posterior_min": p_[m].min(), "posterior_max": p_[m].max()})
det = pd.DataFrame(det)
print(det.pivot(index="dimension", columns="band", values="mean_ci90_width")
      .reindex(columns=[c for c in ["<10", "10-19", "20-49", "50+"] if c in det['band'].unique()])
      .to_string(float_format=lambda v: f"{v:7.3f}"))
print("\nfraction of lenders whose 90% interval is wider than 40 percentage points")
print(det.pivot(index="dimension", columns="band", values="frac_ci_over_40pp")
      .reindex(columns=[c for c in ["<10", "10-19", "20-49", "50+"] if c in det['band'].unique()])
      .to_string(float_format=lambda v: f"{v:7.2f}"))

per-dimension view of the two lowest-volume bands

band                                 <10   10-19   20-49     50+
dimension                                                       
credit_reporting                   0.218   0.154   0.112   0.060
fees_and_costs                     0.448   0.301   0.220   0.137
servicing_and_payment_handling     0.374   0.273   0.213   0.129
unauthorized_or_unrequested_loan   0.250   0.176   0.155   0.083
withdrawal_and_payment_control     0.329   0.213   0.150   0.099

fraction of lenders whose 90% interval is wider than 40 percentage points
band                                 <10   10-19   20-49     50+
dimension                                                       
credit_reporting                    0.00    0.00    0.00    0.00
fees_and_costs                      0.79    0.03    0.00    0.00
servicing_and_payment_handling      0.18    0.00    0.00    0.00
unauthorized_or_unrequested_loan    0.00    0.00    0.00    0.00
withdrawal_and_payment_contro

## 7. Output: `data/processed/payday_shrunk_features.csv`

One row per canonical Payday lender. Contains the complaint count, per-dimension raw counts,
raw shares, posterior means, 90% credible-interval bounds, the fitted prior parameters, and
the outside-the-5-dimensions remainder.

**These are shares, not scores.** Nothing here is normalised across dimensions, weighted, or
ranked.

In [18]:
out = pd.DataFrame({"canonical_company": n.index, "n_complaints": n.values})
out["volume_band"] = out["n_complaints"].map(band)
out["n_outside_dims"] = outside.astype(int)
out["share_outside_dims"] = outside / n_obs

for d in DIMS:
    s = SLUG[d]
    xs = xmat[d].values.astype(float)
    a, b = priors[d]["alpha"], priors[d]["beta"]
    out[f"n_{s}"] = xs.astype(int)
    out[f"share_{s}"] = xs / n_obs
    out[f"post_{s}"] = (xs + a) / (n_obs + a + b)
    out[f"post_lo90_{s}"] = beta_dist.ppf(0.05, xs + a, n_obs - xs + b)
    out[f"post_hi90_{s}"] = beta_dist.ppf(0.95, xs + a, n_obs - xs + b)
    out[f"post_ci90_width_{s}"] = out[f"post_hi90_{s}"] - out[f"post_lo90_{s}"]
    out[f"prior_alpha_{s}"] = a
    out[f"prior_beta_{s}"] = b
    out[f"prior_k_{s}"] = a + b
    out[f"shrink_weight_own_data_{s}"] = n_obs / (n_obs + a + b)

out = out.sort_values("n_complaints", ascending=False).reset_index(drop=True)
out.to_csv(CSV, index=False)
print(f"wrote {CSV}")
print(f"rows {len(out):,}  cols {out.shape[1]}")
print(f"column groups: 1 id + 3 meta + 8 x (count, share, post, lo90, hi90, ci_width, alpha, beta, k, weight)")
print("\nhead (largest lenders)")
print(out.head(6).to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

wrote /Users/mihirkale/cdc/data/processed/payday_shrunk_features.csv
rows 482  cols 55
column groups: 1 id + 3 meta + 8 x (count, share, post, lo90, hi90, ci_width, alpha, beta, k, weight)

head (largest lenders)
                       canonical_company  n_complaints volume_band  n_outside_dims  share_outside_dims  n_withdrawal  share_withdrawal  post_withdrawal  post_lo90_withdrawal  post_hi90_withdrawal  post_ci90_width_withdrawal  prior_alpha_withdrawal  prior_beta_withdrawal  prior_k_withdrawal  shrink_weight_own_data_withdrawal  n_fees  share_fees  post_fees  post_lo90_fees  post_hi90_fees  post_ci90_width_fees  prior_alpha_fees  prior_beta_fees  prior_k_fees  shrink_weight_own_data_fees  n_unauthorized  share_unauthorized  post_unauthorized  post_lo90_unauthorized  post_hi90_unauthorized  post_ci90_width_unauthorized  prior_alpha_unauthorized  prior_beta_unauthorized  prior_k_unauthorized  shrink_weight_own_data_unauthorized  n_credit_rep  share_credit_rep  post_credit_rep  post_

## 8. Assertions

Every number quoted above is re-derived here and checked. The list is long on purpose: this
notebook is the evidence for a modelling choice, and the claims should fail loudly if the
inputs or the estimator change.

In [19]:
res = []

def claim(label, got, expected, tol=5e-3):
    if expected is None or (isinstance(expected, float) and np.isnan(expected)):
        res.append((label, got, None, None, "skip"))
        return
    ok = abs(got - expected) <= tol if isinstance(expected, (int, float)) else got == expected
    res.append((label, float(got), float(expected), float(tol), "PASS" if ok else "FAIL"))

# --- provenance / volume
claim("payday complaints", len(pay), 6024, 0)
claim("canonical payday lenders", L, 482, 0)
claim("median lender volume", float(np.median(n_obs)), 2, 0)
claim("share of lenders n<=2", float((n_obs <= 2).mean()), 0.5394, 0.001)
claim("complaints inside 5 dims", int(xmat.values.sum()), 4958, 0)
claim("dimension share of payday", xmat.values.sum() / len(pay), 0.8231, 0.001)

# --- estimator validation
claim("closed-form vs quadrature max abs diff", maxdiff, 0.0, 1e-9)
claim("ground truth k=20 recovered", float(rec.loc[rec.true_k == 20, "mle_k_median"].iloc[0]), 20.0, 3.0)
claim("ground truth k=10 recovered", float(rec.loc[rec.true_k == 10, "mle_k_median"].iloc[0]), 10.0, 3.0)
claim("ground truth k=30 recovered", float(rec.loc[rec.true_k == 30, "mle_k_median"].iloc[0]), 30.0, 5.0)
claim("MoM underestimates k vs MLE", float((rec["mom_k"] < rec["mle_k_median"]).all()), 1.0, 0)

# --- fitted priors
for d, kk, mm in [("withdrawal_and_payment_control", 4.67, 0.1191),
                  ("fees_and_costs", 5.55, 0.2399),
                  ("unauthorized_or_unrequested_loan", 12.94, 0.1151),
                  ("credit_reporting", 8.63, 0.0707),
                  ("servicing_and_payment_handling", 11.19, 0.2562)]:
    claim(f"prior k :: {SLUG[d]}", priors[d]["k"], kk, 0.05)
    claim(f"prior mean :: {SLUG[d]}", priors[d]["mean"], mm, 0.0005)
    claim(f"prior mean positive & <1 :: {SLUG[d]}",
          float(0 < priors[d]["mean"] < 1), 1.0, 0)

# --- shrinkage behaviour
claim("all zero-count posteriors > 0", float((z[[f"post_{SLUG[d]}" for d in DIMS]] > 0).all().all()), 1.0, 0)
claim("posterior at x=n=1 is < 0.75 for all dims",
      float(p100.iloc[0][[f"post_{SLUG[d]}" for d in DIMS]].max()), 0.75, 1.0)
claim("posterior decreases with n when x=0",
      float((np.diff(z[[f"post_{SLUG[d]}" for d in DIMS]].values, axis=0) < 0).all()), 1.0, 0)
claim("ci width shrinks as n grows (fees)",
      float(np.diff(grid["ci90w_fees"].values).mean() < 0), 1.0, 0)

# --- stability
claim("split-half disagreement reduced", split["disagree_post"].mean() < split["disagree_raw"].mean(), 1.0, 0)
claim("split-half mean raw disagreement", float(split["disagree_raw"].mean()), 0.0721, 0.002)
claim("split-half mean posterior disagreement", float(split["disagree_post"].mean()), 0.0491, 0.002)
claim("split-half reduction pct", (1 - split["disagree_post"].mean() / split["disagree_raw"].mean()) * 100, 31.9, 1.0)
claim("split-half spearman improves every dim",
      float((agree["spearman_post"] > agree["spearman_raw"]).all()), 1.0, 0)
claim("bootstrap posterior sd < raw sd in every dim x band", float(n_worse == 0), 1.0, 0)
claim("bootstrap median sd ratio (non-degenerate)", float(boot["sd_ratio"].median()), 0.5949, 0.02)
claim("noise explains ~all raw-share variance (median pct)", float(nv["noise_share_of_raw_var_pct"].median()), 92.0, 2.0)
claim("posterior sampling variance < raw sampling variance, every dim",
      float((nv["mean_sampling_var_post"] < nv["mean_sampling_var_raw"]).all()), 1.0, 0)

# --- bands
claim("lenders in <10 band", int(bands.loc[bands.band == "<10", "lenders"].iloc[0]), 382, 0)
claim("lenders in 10-19 band", int(bands.loc[bands.band == "10-19", "lenders"].iloc[0]), 30, 0)
claim("lenders in 20-49 band", int(bands.loc[bands.band == "20-49", "lenders"].iloc[0]), 38, 0)
claim("lenders in 50+ band", int(bands.loc[bands.band == "50+", "lenders"].iloc[0]), 32, 0)
claim("band lenders sum to total", int(bands["lenders"].sum()), L, 0)
claim("band complaints sum to total", int(bands["complaints"].sum()), len(pay), 0)
claim("lenders under 20 complaints", int(bands.loc[bands.band.isin(["<10", "10-19"]), "lenders"].sum()), 412, 0)
claim("lenders under 20 is the large majority",
      float(bands.loc[bands.band.isin(["<10", "10-19"]), "lenders"].sum() / L), 0.855, 0.005)
claim("ci90 width shrinks monotonically with volume",
      float((np.diff(bands["mean_ci90_width"].values) < 0).all()), 1.0, 0)
claim("mean ci90 width, <10 band", float(bands.loc[bands.band == "<10", "mean_ci90_width"].iloc[0]), 0.324, 0.005)
claim("mean ci90 width, 50+ band", float(bands.loc[bands.band == "50+", "mean_ci90_width"].iloc[0]), 0.102, 0.005)
claim("no cell has ci>40pp in 20-49 or 50+",
      float(bands.loc[bands.band.isin(["20-49", "50+"]), "frac_ci_over_40pp"].max()), 0.0, 0)
claim("lenders 1-3 complaints", int(corr.loc[corr.band == "1-3", "lenders"].iloc[0]), 331, 0)
claim("lenders 5-9 complaints", int(corr.loc[corr.band == "5-9", "lenders"].iloc[0]), 51, 0)
claim("sub-20 lenders are the large majority but a minority of complaints",
      float(bands.loc[bands.band.isin(["<10", "10-19"]), "complaints"].sum() / len(pay)), 0.218, 0.005)

# --- csv integrity
rt = pd.read_csv(CSV)
claim("csv rows", len(rt), L, 0)
claim("csv n_complaints reconciles", int(rt["n_complaints"].sum()), len(pay), 0)
share_cols = [f"share_{SLUG[d]}" for d in DIMS]
post_cols = [f"post_{SLUG[d]}" for d in DIMS]
claim("csv raw shares that are exactly 0", int((rt[share_cols] == 0).sum().sum()), 1476, 0)
claim("csv raw shares that are exactly 1", int((rt[share_cols] == 1).sum().sum()), 167, 0)
claim("csv posterior shares that are 0 or 1", int(((rt[post_cols] == 0) | (rt[post_cols] == 1)).sum().sum()), 0, 0)
claim("lenders n>1 showing a literal 100% raw share",
      int((rt.loc[rt.n_complaints > 1, share_cols].max(axis=1) == 1.0).sum()), 25, 0)
claim("lenders n>1 showing a literal 0% raw share",
      int((rt.loc[rt.n_complaints > 1, share_cols].min(axis=1) == 0.0).sum()), 244, 0)

for d in DIMS:
    s = SLUG[d]
    claim(f"csv raw counts reconcile :: {s}", int(rt[f"n_{s}"].sum()), int(xmat[d].sum()), 0)
    claim(f"csv posterior strictly inside (0,1) :: {s}", float(rt[f"post_{s}"].between(0, 1, inclusive="neither").all()), 1.0, 0)
    claim(f"csv ci brackets posterior :: {s}",
          float(((rt[f"post_lo90_{s}"] <= rt[f"post_{s}"]) &
                 (rt[f"post_{s}"] <= rt[f"post_hi90_{s}"])).all()), 1.0, 0)
    # posterior must be exactly the weighted blend of raw share and prior mean,
    # and must lie between them (shrinkage never overshoots past the prior)
    a_, b_ = rt[f"prior_alpha_{s}"], rt[f"prior_beta_{s}"]
    k_, m_ = a_ + b_, a_ / (a_ + b_)
    w_ = rt[f"shrink_weight_own_data_{s}"]
    claim(f"posterior == w*raw + (1-w)*prior :: {s}",
          float(np.allclose(w_ * rt[f"share_{s}"] + (1 - w_) * m_, rt[f"post_{s}"], atol=1e-12)), 1.0, 0)
    claim(f"posterior lies between raw and prior :: {s}",
          float(((rt[f"post_{s}"] >= np.minimum(rt[f"share_{s}"], m_) - 1e-12) &
                 (rt[f"post_{s}"] <= np.maximum(rt[f"share_{s}"], m_) + 1e-12)).all()), 1.0, 0)
    # n/(n+k) is monotone but saturating, so rank correlation is the right check
    claim(f"shrink weight monotone in volume :: {s}",
          float(rt["n_complaints"].corr(rt[f"shrink_weight_own_data_{s}"], method="spearman") > 0.999), 1.0, 0)
    claim(f"shrink weight == n/(n+k) :: {s}",
          float(np.allclose(rt[f"shrink_weight_own_data_{s}"],
                            rt["n_complaints"] / (rt["n_complaints"] + k_), atol=1e-12)), 1.0, 0)
claim("outside-dims reconciles", int(rt["n_outside_dims"].sum()), len(pay) - int(xmat.values.sum()), 0)
claim("csv has no score or rank column",
      float(not any(k for k in rt.columns
                    if any(t in k.lower() for t in ("score", "rank", "weight_total", "grade")))), 1.0, 0)

chk = pd.DataFrame(res, columns=["claim", "got", "expected", "tol", "status"])
chk["got"] = chk["got"].round(6)
print(chk.to_string(index=False))
n_fail = int((chk["status"] == "FAIL").sum())
n_skip = int((chk["status"] == "skip").sum())
print(f"\n{(chk['status'] == 'PASS').sum()}/{len(chk)} claims verified"
      f"   ({n_skip} skipped, {n_fail} FAILED)")
if n_fail:
    print("\nFAILED CLAIMS:")
    print(chk[chk["status"] == "FAIL"].to_string(index=False))
assert n_fail == 0, f"{n_fail} claims failed"

                                                             claim         got  expected          tol status
                                                 payday complaints 6024.000000 6024.0000 0.000000e+00   PASS
                                          canonical payday lenders  482.000000  482.0000 0.000000e+00   PASS
                                              median lender volume    2.000000    2.0000 0.000000e+00   PASS
                                             share of lenders n<=2    0.539419    0.5394 1.000000e-03   PASS
                                          complaints inside 5 dims 4958.000000 4958.0000 0.000000e+00   PASS
                                         dimension share of payday    0.823041    0.8231 1.000000e-03   PASS
                            closed-form vs quadrature max abs diff    0.000000    0.0000 1.000000e-09   PASS
                                       ground truth k=20 recovered   22.381435   20.0000 3.000000e+00   PASS
                   

## 9. Answers

### 1. Does empirical-Bayes shrinkage materially improve stability?

**Yes, and the improvement is quantified rather than illustrative.**

- **Split-half.** Across the 70 Payday lenders with at least 20 complaints, 200 random splits
  each: the mean absolute disagreement between the two halves falls from **0.072** on raw
  shares to **0.049** on posterior means, a **32% reduction**. Half-to-half rank correlation
  improves for **all five** dimensions, e.g. credit reporting 0.53 -> 0.70 and unauthorized
  loans 0.47 -> 0.57.
- **Bootstrap.** Resampling each lender's complaints 2,000 times, the posterior sampling SD is
  below the raw sampling SD in **20 of 20** dimension x volume-band combinations. The ratio is
  0.25-0.44 below 10 complaints and rises to 0.87-0.95 at 50+, i.e. shrinkage helps most
  exactly where the data are weakest and correctly backs off where they are strong.
- **Decomposition.** A median **92%** of the cross-lender variance in raw shares (range
  79-101%) is pure sampling noise. Raw per-lender shares are largely measuring how many
  complaints a lender happens to have filed, not how that lender behaves.
- **Degeneracy.** 1,476 of 2,410 lender x dimension cells have a raw share of *exactly* 0% and
  167 have *exactly* 100%. Twenty-five lenders with more than one complaint display a literal
  100% share on some dimension. No posterior share is ever 0% or 100%.

### 2. What prior strength does the data imply?

The fitted effective prior sample sizes $k_d = \alpha_d + \beta_d$ are small:

| dimension | k | prior mean | pooled peer prevalence |
|---|---|---|---|
| withdrawal_and_payment_control | ~4.7 | ~0.12 | ~0.15 |
| fees_and_costs | ~5.5 | ~0.24 | ~0.30 |
| unauthorized_or_unrequested_loan | ~12.9 | ~0.12 | ~0.10 |
| credit_reporting | ~8.6 | ~0.07 | ~0.06 |
| servicing_and_payment_handling | ~11.2 | ~0.26 | ~0.22 |

This is a real and somewhat counter-intuitive result. The most differentiated dimension
(withdrawal) gets the **weakest** prior, $k \\approx 4.7$, because lender rates genuinely vary
a lot there, so the data earn a low prior strength. Conversely, unauthorized loans get the
strongest prior, $k \\approx 12.9$, because lenders look genuinely similar on that dimension.

Consequence: the practical meaning of $k$ is that a lender needs on the order of 5 to 13
complaints before its own data dominate the peer prior. A lender with 5 complaints is roughly
half prior, half own data for every dimension here.

The prior mean sits near the lender-weighted mean of per-lender rates, not near the pooled
complaint-weighted prevalence. That is correct behaviour, and it matters: pooled prevalence is
dominated by large lenders, and using it as the anchor for a small lender would bias small
lenders toward the behaviour of big ones.

### 3. How much does shrinkage affect lenders at different sample sizes?

| band | lenders | complaints | mean abs correction | raw sd -> posterior sd | mean 90% CI width | CI wider than 40pp |
|---|---|---|---|---|---|---|
| <10 | 382 | 900 | large | 0.29 -> 0.05 | 0.324 | 22% of cells |
| 10-19 | 30 | 416 | moderate | 0.14 -> 0.09 | 0.223 | 1% of cells |
| 20-49 | 38 | 1,107 | small | 0.14 -> 0.11 | 0.170 | 0% |
| 50+ | 32 | 3,601 | small | 0.13 -> 0.12 | 0.102 | 0% |

- **1-4 complaints (331 lenders):** heavily shrunk; these lenders mostly report the peer prior.
  A lender whose only complaint was a withdrawal shows a small positive withdrawal rate rather
  than 100%, and a lender with zero withdrawal complaints still shows a small positive rate.
- **5-9 (51 lenders):** still dominated by the prior; the posterior is roughly a blend.
- **10-19 (30 lenders):** comparable to the prior strength for most dimensions, so the estimate
  begins to reflect the lender's own record. Cross-lender dispersion is still 36% above the
  shrunken level.
- **20-49 (38 lenders):** the lender's own data clearly dominate.
- **50+ (32 lenders):** the posterior is close to the raw share; shrinkage is a correction of
  a few points (mean absolute move under 0.015) rather than a reinterpretation.

The 90% credible-interval width falls monotonically with volume, from 0.324 on average below
10 complaints to 0.102 at 50+, and the fraction of cells with an interval wider than 40
percentage points goes 22% -> 1% -> 0% -> 0%. That is the honest companion to the point
estimate.

### 4. Can FinePrint display estimates for low-volume lenders with an explicit confidence indicator, or is a hard minimum still necessary?

**A hard minimum is not strictly necessary, and shrinkage plus a confidence interval is the
better design, with one caveat.**

**Why the usual minimum exists.** A volume floor is normally imposed to stop exactly the
failures documented above: 61% of lender x dimension cells have a raw share of exactly 0%, 7%
have exactly 100%, and 25 lenders with more than one complaint show a literal 100% share.
Those are artefacts of a denominator of 2 or 3, not findings.

**Why the posterior handles all of it.** By construction, no posterior share is ever exactly 0
or exactly 1, the extreme values compress automatically as $k$ and $n$ change, and the
posterior is always a convex combination of the lender's own share and the peer prior, so it
can never overshoot past the prior. A 90% interval is available for every lender and dimension
with no extra modelling, and the 40-percentage-point ambiguity that motivates suppression
disappears above 20 complaints (0% of cells in the 20-49 and 50+ bands).

**The caveat.** Shrinkage buys honesty, not precision. 412 of 482 lenders (85.5%) have fewer
than 20 complaints, and although they account for only 21.8% of complaints, their estimates
remain wide: 22% of sub-20 cells have an interval wider than 40 percentage points. An interval
is therefore genuinely necessary rather than cosmetic.

Two design implications follow:

- Display the posterior share together with its interval width, or a coarse
  low/medium/high-volume indicator, so a weak number is visibly weak.
- Prefer not to sort or rank lenders whose intervals overlap heavily. Shrinkage makes small
  lenders comparable with large ones, but heavily overlapping intervals still mean the
  ordering is not meaningful. Ranking is out of scope here by instruction.

No product threshold is chosen here. If one is needed later it should be a **display** rule
about how prominently to show low-volume estimates, not a rule that deletes lenders from the
data.